### 安装依赖

In [ ]:
!pip install -U ragas arxiv pymupdf wandb tiktoken

In [2]:
from getpass import getpass


### 数据准备

主要以 Arxiv 论文为例进行评估，通过 `ArxivLoader` 加载 Arxiv 论文数据

In [1]:
from langchain_community.document_loaders import ArxivLoader

c:\Users\player\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
paper_docs = ArxivLoader(query="2309.15217").load()
len(paper_docs)

1

In [3]:
len(paper_docs[0].page_content)


31945

In [4]:
for doc in paper_docs:
    print(doc.metadata)

{'Published': '2025-04-28', 'Title': 'Ragas: Automated Evaluation of Retrieval Augmented Generation', 'Authors': 'Shahul Es, Jithin James, Luis Espinosa-Anke, Steven Schockaert', 'Summary': 'We introduce Ragas (Retrieval Augmented Generation Assessment), a framework for reference-free evaluation of Retrieval Augmented Generation (RAG) pipelines. RAG systems are composed of a retrieval and an LLM based generation module, and provide LLMs with knowledge from a reference textual database, which enables them to act as a natural language layer between a user and textual databases, reducing the risk of hallucinations. Evaluating RAG architectures is, however, challenging because there are several dimensions to consider: the ability of the retrieval system to identify relevant and focused context passages, the ability of the LLM to exploit such passages in a faithful way, or the quality of the generation itself. With Ragas, we put forward a suite of metrics which can be used to evaluate these

### 文本分割、Embedding model、向量数据库

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
)

docs = text_splitter.split_documents(paper_docs)
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=DashScopeEmbeddings(model="text-embedding-v4")
)
len(docs)


107

In [9]:
max([len(doc.page_content) for doc in docs])

497

利用 chroma 向量库的 `.as_retriever()` 方式进行检索，需要控制的主要参数为 `k`，即返回的检索结果数量。

In [10]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [11]:
relevant_docs = base_retriever.invoke("What is Retrieval Augmented Generation?")
len(relevant_docs)

3

### 创建 prompt 生成答案

我们需要利用 `LLM` 对 `Context` 生成一系列问题的 `Answer`。

In [15]:
from langchain_core.prompts import PromptTemplate

template = """You are an assistant for question-answering task.
Use the following pieces of retrieved context to answer the quesion.
If you don't know the answer, just say that you don't know.

Question: {question}

Context: {context}

Answer:
"""

prompt = PromptTemplate.from_template(
    template = template,
)
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template="You are an assistant for question-answering task.\nUse the following pieces of retrieved context to answer the quesion.\nIf you don't know the answer, just say that you don't know.\n\nQuestion: {question}\n\nContext: {context}\n\nAnswer:\n"


### 利用 LLM 生成答案

In [16]:
from langchain_community.chat_models import ChatTongyi
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatTongyi(model="qwen-plus") # type: ignore

rag_chain = (
    {"context": base_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### 创建 RAGAS 所需的数据

Ragas 数据格式要求 ['question', 'answer', 'contexts', 'ground_truths']
```json
{
    "question": [],     // 基于 `Context` 生成的问题列表
    "answer": [],       // llm 生成的答案
    "contexts": [],     // context
    "ground_truths": [] // 标准答案
}
```


In [21]:
from datasets import Dataset

questions = [
  "what is faithfulness ?",
  "How many pages are included in the WikiEval dataset, and which years do they cover information from?",
  "why is evaluating Retrieval Augmented Generation (RAG) systems challenging?"
]
ground_truths = [
  ["Faithfulness refers to the idea that the answer should be grounded in the given context."],
  ["To construct the dataset, we first selected 50 Wikipedia pages covering events that happened since the start of 2022."],
  ["Evaluating RAG architectures is, however, challenging because there are several dimensions to consider: the ability of the retrieval system to identify relevant and focused content and the ability of the generative model to ground its answers in the retrieved content."]
]
answers = []
contexts = []

# 生成答案
for query in questions:
    answer = rag_chain.invoke(query)
    answers.append(answer)
    contexts.append([doc.page_content for doc in base_retriever.invoke(query)])

# 构建数据
data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truths": ground_truths
}
dataset = Dataset.from_dict(data)
    


In [ ]:
dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truths'],
    num_rows: 3
})

: 

### 使用 Ragas 进行评估

In [46]:
!pip install litellm

   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
    --------------------------------------- 0.3/15.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/15.6 MB 2.2 MB/s eta 0:00:07
   --- ------------------------------------ 1.3/15.6 MB 1.8 MB/s eta 0:00:09
   ---- ----------------------------------- 1.6/15.6 MB 1.9 MB/s eta 0:00:08
   ----- ---------------------------------- 2.1/15.6 MB 1.9 MB/s eta 0:00:08
   ------ --------------------------------- 2.6/15.6 MB 2.0 MB/s eta 0:00:07
   -------- ------------------------------- 3.4/15.6 MB 2.2 MB/s eta 0:00:06
   ---------- ----------------------------- 4.2/15.6 MB 2.4 MB/s eta 0:00:05
   ------------ --------------------------- 4.7/15.6 MB 2.4 MB/s eta 0:00:05
   -------------- ------------------------- 5.5/15.6 MB 2.6 MB/s eta 0:00:04
   -------------- ------------------------- 5.8/15.6 MB 2.6 MB/s eta 0:00:04
   -----------------

In [63]:
from langchain_community.llms import Tongyi
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    faithfulness, answer_relevancy, context_precision,
    context_recall, answer_correctness
)
from ragas import evaluate


# 初始化评估 LLM
tongyi_llm = Tongyi(model_name="qwen-plus", temperature=0.0)
evaluator_llm = LangchainLLMWrapper(tongyi_llm)

# 评估
result = evaluate(
    dataset=dataset,
    metrics=[context_precision, context_recall, faithfulness, 
             answer_relevancy, answer_correctness],
    llm=evaluator_llm,
)


ValueError: The metric [context_precision] that is used requires the following additional columns ['reference'] to be present in the dataset.